In [ ]:
# Lab type: extend
# Course: ML201 — Applied Machine Learning
# Lesson: Understanding Model Predictions with SHAP
# Task: Start with the baseline code below, then extend it with the exercises provided.

# Lab: Explaining Model Predictions with SHAP

The baseline trains a RandomForestClassifier on a synthetic loan approval dataset (n=2000, features: income, credit_score, loan_amount, debt_ratio, employment_years, num_accounts, region). It then computes SHAP values using `shap.TreeExplainer` and plots a beeswarm summary showing global feature importance. The extensions ask you to dig into individual predictions (waterfall plots), feature interactions (dependence plots), alternative importance metrics (mean absolute SHAP vs MDI), and misclassification analysis.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 2000

income = np.random.lognormal(mean=10.8, sigma=0.5, size=n)  # ~$50k median
credit_score = np.random.normal(680, 80, n).clip(300, 850)
loan_amount = np.random.lognormal(mean=10.3, sigma=0.6, size=n).clip(5000, 500000)
debt_ratio = np.random.beta(2, 5, n)  # 0-1 range
employment_years = np.random.exponential(7, n).clip(0, 40)
num_accounts = np.random.poisson(4, n).clip(0, 20)
region_codes = np.random.choice([0, 1, 2, 3], n, p=[0.3, 0.25, 0.25, 0.2])

log_odds = (
    -2.0
    + 0.0000035 * income
    + 0.008 * (credit_score - 680)
    - 0.0000012 * loan_amount
    - 2.5 * debt_ratio
    + 0.06 * employment_years
    + 0.08 * num_accounts
    + 0.2 * (region_codes == 0).astype(float)
)
prob_approved = 1 / (1 + np.exp(-log_odds))
approved = np.random.binomial(1, prob_approved)

feature_names = ['income', 'credit_score', 'loan_amount', 'debt_ratio',
                 'employment_years', 'num_accounts', 'region']

X = pd.DataFrame({
    'income': income,
    'credit_score': credit_score,
    'loan_amount': loan_amount,
    'debt_ratio': debt_ratio,
    'employment_years': employment_years,
    'num_accounts': num_accounts,
    'region': region_codes.astype(float)
})
y = pd.Series(approved, name='approved')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Test AUC: {test_auc:.3f}")
print(f"Feature names: {feature_names}")
print(f"Test set size: {len(X_test)}")

## Step 1: Compute SHAP Values with TreeExplainer

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values is a list of two arrays: [class_0, class_1]
print(f"shap_values type  : {type(shap_values)}")
print(f"Number of classes : {len(shap_values)}")
print(f"shap_values[1] shape: {shap_values[1].shape}  (test examples x features)")
print(f"Base value (class 1): {explainer.expected_value[1]:.4f}")

# Verify additivity: SHAP values sum to model output for example 0
shap_sum_0 = shap_values[1][0].sum() + explainer.expected_value[1]
model_pred_0 = model.predict_proba(X_test.iloc[[0]])[:, 1][0]
print(f"\nAdditivity check (example 0):")
print(f"  SHAP sum + base value : {shap_sum_0:.4f}")
print(f"  model.predict_proba   : {model_pred_0:.4f}")
print(f"  Match: {abs(shap_sum_0 - model_pred_0) < 1e-6}")

**Note:** The additivity check above confirms that SHAP values exactly reconstruct every individual prediction. Each feature's SHAP value represents its marginal contribution — summing all features' contributions plus the base value (average model output over the training set) reproduces the model's exact probability for that example.

## Step 2: Beeswarm Summary Plot (global)

In [ ]:
shap.summary_plot(shap_values[:, :, 1], X_test, feature_names=feature_names, show=False)
plt.title('SHAP Beeswarm Summary — Loan Approval Model')
plt.tight_layout()
plt.show()

## Extension 1: Waterfall Plot for a Single Prediction

Choose two test examples: one that the model predicts as high probability approved (> 0.8) and one as low probability (< 0.2). For each, create a waterfall plot using `shap.waterfall_plot(shap.Explanation(...))`. Compare the feature contributions side by side to understand which features drive each decision.

In [ ]:
# Find a high-probability and low-probability example
test_probs = model.predict_proba(X_test)[:, 1]

# YOUR CODE HERE: find indices where predict_proba > 0.8 and < 0.2
high_idx = ...  # index into X_test where test_probs > 0.8
low_idx = ...   # index into X_test where test_probs < 0.2

# Plot waterfall for high_idx
# shap.waterfall_plot(shap.Explanation(
#     values=shap_values[1][high_idx],
#     base_values=explainer.expected_value[1],
#     data=X_test.iloc[high_idx],
#     feature_names=feature_names
# ))

# Plot waterfall for low_idx
# shap.waterfall_plot(shap.Explanation(
#     values=shap_values[1][low_idx],
#     base_values=explainer.expected_value[1],
#     data=X_test.iloc[low_idx],
#     feature_names=feature_names
# ))

**Question:** Looking at the two waterfall plots, which features push the prediction in opposite directions for the two examples? What does this tell you about the model's decision boundary?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Extension 1</summary>

**Features that push in opposite directions:** In a loan approval model, the high-probability example will show `credit_score`, `income`, and `employment_years` contributing large positive SHAP values (pushing toward approval), while `debt_ratio` and `loan_amount` contribute negatively but are outweighed. For the low-probability example the signs flip — high `debt_ratio` pushes strongly negative, `credit_score` may be below average and also negative, while `income` may be near-neutral or slightly positive.

**What this tells you about the decision boundary:** The model's boundary is not a single threshold on one feature — it is defined by the interplay of these features. A high `credit_score` is not sufficient on its own if `debt_ratio` is very high; the boundary is effectively a weighted combination. Features that show opposite-signed SHAP values across the two examples are the ones that actually discriminate between approval and rejection.

</details>

## Extension 2: Dependence Plot

Create a dependence scatter plot for `credit_score` with `interaction_index='income'`. The x-axis shows the raw `credit_score` value; the y-axis shows the SHAP value for `credit_score` for each test example; each point is coloured by the applicant's `income`. This reveals how the effect of credit score on the prediction varies depending on income level.

In [ ]:
# shap.dependence_plot(
#     "credit_score",
#     shap_values[1],
#     X_test,
#     interaction_index="income",
#     feature_names=feature_names
# )

**Question:** If you see a cluster of high-income applicants with low `credit_score` but high positive SHAP values for `credit_score`, what would this suggest about the model's interaction between `credit_score` and `income`?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Extension 2</summary>

**What the cluster suggests:** High-income applicants with low `credit_score` but *positive* SHAP values for `credit_score` suggests the model has learned an interaction: when income is high, the model treats even a mediocre credit score more favourably — income compensates for credit score. The SHAP value for `credit_score` is not purely a function of credit score alone; it absorbs part of the joint predictive contribution from the credit_score-income combination that cannot be cleanly attributed to either feature independently.

**Important caveat:** This is what the model learned from the training data. It may reflect a genuine pattern (higher-income borrowers with lower scores are still lower-risk) or a training data artifact. The dependence plot reveals the model's behaviour, not a causal truth.

</details>

## Extension 3: Mean Absolute SHAP as Global Importance

Compute the mean absolute SHAP value for each feature (the global importance ranking) and plot it as a horizontal bar chart. Compare this ranking to MDI (Mean Decrease in Impurity) from `rf.feature_importances_`. Are the rankings the same? SHAP importance measures average prediction impact; MDI measures impurity reduction during training — they can diverge, especially for features correlated with others.

In [ ]:
# mean_abs_shap = ...  # shape (n_features,): mean over test examples of |shap_values[1]|

# Create a DataFrame with columns ['feature', 'mean_abs_shap', 'mdi_importance']
# Plot side by side as a grouped bar chart or two subplots

# YOUR CODE HERE

**Question:** If `region` appears higher in SHAP importance than in MDI importance, what does this suggest about `region`'s predictive contribution versus its impurity reduction role in tree splits?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Extension 3</summary>

**What the divergence suggests:** `region` appearing higher in mean absolute SHAP than in MDI suggests the feature contributes meaningfully to the model's *probability estimates* across test examples, even though its impurity-reduction contribution during training was modest. MDI underestimates discrete features with few unique values (region has only 4) because each split on `region` divides the data into at most 4 groups — far fewer splits than a continuous feature like `income` that can create thousands of threshold splits, each reducing impurity. SHAP captures the actual probability shift per prediction, which is unaffected by feature cardinality.

</details>

## Extension 4 (Challenge): Explain a Misclassification

Find a test example that the model classified incorrectly (predicted approved probability > 0.5 but true label is 0, or predicted < 0.5 but true label is 1). Create a waterfall plot for it and identify which features pushed the model toward the wrong prediction. This is a common real-world debugging workflow.

In [ ]:
# Find misclassified examples
y_pred = (model.predict_proba(X_test)[:, 1] > 0.5).astype(int)
y_true = y_test.values if hasattr(y_test, 'values') else y_test

# misclassified = ... find indices where y_pred != y_true
# Pick one misclassified example and plot its SHAP waterfall

# YOUR CODE HERE

**Question:** Is the waterfall plot for a misclassification informative for debugging the model, or does it only explain what the model did (not why it was wrong)? What would you do next to investigate?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Extension 4</summary>

**The waterfall only explains the model, not why it was wrong.** It shows which features pushed the prediction in the wrong direction — but "wrong" is defined relative to the true label, which the model never saw at inference time. The waterfall cannot tell you whether the model's logic is flawed or whether this example is genuinely ambiguous (an outlier in the training distribution, a noisy label, or a rare combination of features).

**What to do next:** (1) Check whether the misclassified example falls in a sparse region of the training feature space — if no similar examples existed during training, the model is extrapolating. (2) Examine multiple misclassifications with the same predicted direction to identify systematic patterns (same features, same directions). (3) Look at the feature values: are they plausible, or does the example have an unusual combination that the model has never seen? (4) Investigate whether the true label might itself be noisy (mislabelled training or test example).

</details>

## Summary

Answer these final check questions in one sentence each:

1. `TreeExplainer` is fast because it exploits tree structure. If you needed to explain a neural network instead, which SHAP explainer would you use, and what is the main trade-off?

2. The additivity property guarantees that SHAP values sum to the model's prediction for every example. Why does this make SHAP more trustworthy than permutation feature importance for explaining individual predictions?

3. Two features (`income` and `loan_amount`) are highly correlated. How does this affect the interpretation of their individual SHAP values?

4. A SHAP dependence plot shows that `credit_score` has a strong positive effect. Can you conclude from this that improving a customer's credit score will cause their approval probability to increase? Why or why not?

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Neural network explainer:** Use `shap.KernelExplainer` (model-agnostic) or `shap.DeepExplainer` (PyTorch/TensorFlow-specific) — the main trade-off is speed: `TreeExplainer` exploits the tree structure for exact polynomial-time computation, while `KernelExplainer` uses Monte Carlo sampling of feature coalitions, which can be orders of magnitude slower and only approximates the SHAP values.

2. **Why SHAP additivity makes it more trustworthy for individual predictions:** SHAP's additivity guarantee anchors each value to a specific, exact prediction — the sum of all SHAP values plus the base value equals the model's output for that exact example. Permutation feature importance is a global aggregate over the entire dataset and cannot decompose a single prediction; if you want to understand why the model approved or rejected one specific applicant, only SHAP (or similar local methods) provides a valid attribution.

3. **Correlated features and SHAP interpretation:** Because `income` and `loan_amount` are correlated, their individual SHAP values are partially confounded — removing income's "marginal contribution" while holding `loan_amount` constant creates counterfactuals that may not exist in the data distribution. The SHAP values reflect the Shapley-value game-theoretic attribution (average marginal contribution over all possible feature orderings), which distributes shared predictive power between correlated features but does not cleanly isolate causal contributions.

4. **Correlation vs causation in SHAP dependence plots:** A SHAP dependence plot shows the model's learned association between `credit_score` and its predicted output — it cannot be interpreted causally. Credit score may proxy for other variables (payment history, employment stability, loan-to-income ratio) that were also present in training data; improving a customer's credit score number does not guarantee the same change in approval probability if the underlying correlates remain unchanged.

</details>